In [4]:
pip install statsmodels --user


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
import matplotlib.patches as mpatches
from matplotlib.ticker import PercentFormatter

# =========================
# Paths (adjust if needed)
# =========================
super_path = "/Users/judycheng/Desktop/supercharger in washington state.xls"
residents_path = "/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx"
ev_path = "/Users/judycheng/Desktop/coordinates_output.xlsm"

# =========================
# Step 1: Read Excel files
# =========================
super_df = pd.read_excel(super_path)
residents_df = pd.read_excel(residents_path)
ev_df = pd.read_excel(ev_path)

# =================================
# Step 2: Filter to King County data
# =================================
king_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "King County")]
king_residents = residents_df[residents_df["County"] == "King County"]
king_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "King")]

population_column = "total"
population_25_59 = float(king_residents[population_column].values[0])
num_chargers = int(len(king_super))
num_evs = int(len(king_ev))

print(f"King County population (25-59): {population_25_59:,.0f}")
print(f"Existing Superchargers: {num_chargers}")
print(f"Existing EVs (rows in EV file for King): {num_evs}")

# ==========================
# Step 3: Policy targets
# ==========================
policy_targets = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}

# =================================================
# Step 4: Charger target math
# =================================================
king_county_area = 2307  # sq mi
radius_miles = 2
area_per_charger = np.pi * radius_miles**2
lower_chargers_goal = int(np.ceil(king_county_area / area_per_charger))  # 2-mile geo-coverage
upper_chargers_goal = int(round(population_25_59 / 1500.0))              # 1 per 1,500 residents

print(f"Lower bound chargers (geo coverage): {lower_chargers_goal}")
print(f"Upper bound chargers (1 per 1,500 residents): {upper_chargers_goal}")

# ====================
# Step 5: Build scenarios
# ====================
years = list(range(2025, 2051))
n_years = len(years)
rows = []

# ---- Lower: slow Phase 2, catch-up Phase 3 ----
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers_goal - num_chargers)
yearly_build_lower = int(round(total_lower_needed / n_years)) if n_years > 0 else 0

for y in years:
    # Chargers build (constant toward geo cap)
    new_lower = yearly_build_lower
    if current_lower + new_lower > lower_chargers_goal:
        new_lower = lower_chargers_goal - current_lower
    current_lower += max(0, new_lower)

    # Adoption: P1 modest; P2 slower than Upper; P3 catch-up
    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5) * 0.35
    elif y <= 2040:
        eff_start = 0.10 + (policy_targets[2030] - 0.10) * 0.35
        eff = eff_start + (policy_targets[2040] * 0.70 - eff_start) * ((y - 2030) / 10)
    else:
        eff_start = policy_targets[2040] * 0.70
        eff = eff_start + (policy_targets[2050] - eff_start) * ((y - 2040) / 10)

    rows.append({
        "Scenario": "Lower",
        "Year": y,
        "Total_Chargers": int(current_lower),
        "New_Chargers": int(max(0, new_lower)),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

# ---- Upper: 3-phase slowdown, aggressive to meet targets ----
current_upper = num_chargers
total_upper_needed = max(0, upper_chargers_goal - num_chargers)

# Allocate adds by phase (front-load then taper)
phase_weights = []
for y in years:
    if 2025 <= y <= 2030: phase_weights.append(1.0)
    elif 2031 <= y <= 2040: phase_weights.append(0.6)
    else: phase_weights.append(0.2)
phase_weights = np.array(phase_weights)
phase_weights = phase_weights / phase_weights.sum()
yearly_adds = np.round(phase_weights * total_upper_needed).astype(int)

# Ensure exact sum
diff = total_upper_needed - yearly_adds.sum()
if diff != 0:
    idx = 0
    step = 1 if diff > 0 else -1
    for _ in range(abs(diff)):
        yearly_adds[idx] += step
        idx = (idx + 1) % len(yearly_adds)

for y, add in zip(years, yearly_adds):
    current_upper += max(0, int(add))

    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5)
    elif y <= 2035:
        eff = policy_targets[2030] + (policy_targets[2035] - policy_targets[2030]) * ((y - 2030) / 5)
    elif y <= 2040:
        eff = policy_targets[2035] + (policy_targets[2040] - policy_targets[2035]) * ((y - 2035) / 5)
    else:
        eff = policy_targets[2040] + (policy_targets[2050] - policy_targets[2040]) * ((y - 2040) / 10)

    rows.append({
        "Scenario": "Upper",
        "Year": y,
        "Total_Chargers": int(current_upper),
        "New_Chargers": int(max(0, int(add))),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

sc_df = pd.DataFrame(rows)

# ===============================
# Step 6: Monte Carlo (Monotone; adoption responds to charger buildup)
# ===============================
yrs = sc_df["Year"].unique()
lo = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Lower"].values
hi = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Upper"].values

# --- 6A) Target path the MC reverts toward (NOT midpoint)
target_w_upper = []
for y in yrs:
    if 2025 <= y <= 2030:   target_w_upper.append(0.55)  # slightly neutral early
    elif 2031 <= y <= 2035: target_w_upper.append(0.25)  # lean low in early P2
    else:                   target_w_upper.append(0.70)  # accelerate after 2035
target_w_upper = np.array(target_w_upper, dtype=float)

raw_target = (1 - target_w_upper) * lo + target_w_upper * hi
target_cap = 0.94  # keep MC under 95% by 2050; tweak if desired
target = np.minimum(raw_target, target_cap)
target = np.maximum.accumulate(target)  # ensure target itself non-decreasing

# --- 6B) Public charger growth & home-charger dampening
w_upper_charger = []
for y in yrs:
    if 2025 <= y <= 2030:   w_upper_charger.append(0.80)
    elif 2031 <= y <= 2040: w_upper_charger.append(0.30)
    else:                   w_upper_charger.append(0.55)
w_upper_charger = np.array(w_upper_charger, dtype=float)

lowC = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Lower"].values
hiC  = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Upper"].values

# (NEW) compute a blended charger path, then ENFORCE monotonicity
C_blend = ((1 - w_upper_charger) * lowC + w_upper_charger * hiC).round().astype(int)
C_blend = np.maximum.accumulate(C_blend)  # <-- monotonic chargers
assert np.all(np.diff(C_blend) >= 0), "Internal: blended chargers decreased unexpectedly."

# Convert to growth metrics
C_growth = np.r_[0, np.diff(C_blend)]
den = np.maximum(1.0, np.r_[C_blend[0], C_blend[:-1]])
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den>0)
C_growth_rate = np.clip(C_growth_rate, 0, 0.50)

# Home-charger share (dampens public network’s marginal effect)
home_share = []
for y in yrs:
    if 2025 <= y <= 2030:   home_share.append(0.55 + (0.72 - 0.55) * (y - 2025) / 5)
    elif 2031 <= y <= 2040: home_share.append(0.72 + (0.90 - 0.72) * (y - 2030) / 10)
    else:                   home_share.append(0.90 + (0.93 - 0.90) * (y - 2040) / 10)
home_share = np.array(home_share, dtype=float)

# (NEW) Momentum of charger build-out to amplify coupling in high-growth years
# Simple 3-year trailing avg of growth rate (with padding)
trail = 3
pad = np.full(trail-1, C_growth_rate[1])  # repeat first nonzero-ish growth
C_gr_trailing = np.convolve(np.r_[pad, C_growth_rate], np.ones(trail)/trail, mode="valid")
C_gr_trailing = C_gr_trailing[:len(C_growth_rate)]
charger_momentum = np.clip(C_gr_trailing, 0.0, 0.40)  # cap to avoid runaway

# ======================
# 6C) Monotone MC in rate space (adoption responds to growth & momentum)
# ======================
N = 1000
T = len(yrs)
rng = np.random.default_rng(42)

# Phase parameters: baseline values
kappa_base = np.zeros(T)
beta_base  = np.zeros(T)
sigma      = np.zeros(T)
for i, y in enumerate(yrs):
    if 2025 <= y <= 2030:
        kappa_base[i] = 0.30; beta_base[i] = 0.45; sigma[i] = 0.015
    elif 2031 <= y <= 2035:
        kappa_base[i] = 0.25; beta_base[i] = 0.20; sigma[i] = 0.010
    elif 2036 <= y <= 2040:
        kappa_base[i] = 0.45; beta_base[i] = 0.30; sigma[i] = 0.018
    else:
        kappa_base[i] = 0.55; beta_base[i] = 0.30; sigma[i] = 0.020

# (NEW) Adaptive coupling to chargers:
# beta_effect = beta_base * (1 - home_share) * (1 + 1.5*charger_momentum)
beta_effect = beta_base * (1 - home_share) * (1 + 1.5 * charger_momentum)
beta_effect = np.clip(beta_effect, 0.0, 0.9)  # keep stable

paths = np.zeros((N, T), dtype=float)
start_level = float(np.clip(0.35 * lo[0] + 0.65 * hi[0], 0.0, target_cap))  # start upper-leaning but capped
paths[:, 0] = start_level

for t in range(1, T):
    prev = paths[:, t-1]
    mean_revert = kappa_base[t] * (target[t] - prev)     # >0 if below target
    charger_push = beta_effect[t] * C_growth_rate[t]     # >= 0, amplified by momentum & dampened by home share
    noise = rng.normal(0.0, sigma[t], size=N)            # small, zero-mean
    delta = mean_revert + charger_push + noise
    delta = np.maximum(delta, 0.0)                       # forbid negative increments
    next_rate = prev + delta
    next_rate = np.maximum(next_rate, lo[t])             # never below Lower for that year
    next_rate = np.minimum(next_rate, target_cap)        # keep <95%
    next_rate = np.maximum(next_rate, prev)              # enforce monotonicity
    paths[:, t] = next_rate

# Percentiles per year
p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p90 = np.percentile(paths, 90, axis=0)

ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# ===========================================
# Step 7: Build final table with new columns
# ===========================================
wide = sc_df.pivot(index="Year", columns="Scenario", values=["Total_Chargers", "Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

# (NEW) Monotonic, budget-weighted charger path as Forecast_Chargers
budget_weighted_C = C_blend.copy()
assert np.all(np.diff(budget_weighted_C) >= 0), "Forecast chargers decreased — check weighting!"
wide["Forecast_Chargers"] = budget_weighted_C

# MC outputs
wide["Forecast_Adoption_P10"] = p10
wide["Forecast_Adoption_P50"] = p50
wide["Forecast_Adoption_P90"] = p90
wide["Forecast_EVs_P10"] = ev_p10
wide["Forecast_EVs_P50"] = ev_p50
wide["Forecast_EVs_P90"] = ev_p90

# Optional check: how often P50 sits within [Lower, Upper]
within_band = ((wide["Forecast_Adoption_P50"] >= wide["Adoption_Rate_Lower"]) &
               (wide["Forecast_Adoption_P50"] <= wide["Adoption_Rate_Upper"]))
wide["Forecast_within_bounds"] = within_band.astype(int)

# =================================
# Step 8: Save results to Excel
# =================================
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "king_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path, engine="openpyxl") as writer:
    sc_df.to_excel(writer, sheet_name="Scenarios_LU", index=False)
    wide.to_excel(writer, sheet_name="Forecast", index=False)

# ==============================
# Step 9: Charts (with phases)
# ==============================
phase_spans = [(2025, 2030, "Phase 1"), (2030, 2040, "Phase 2"), (2040, 2050, "Phase 3")]
phase_colors = ["green", "yellow", "orange"]

# --- A) Total Chargers: Lower vs Upper vs Budget-weighted Forecast ---
plt.figure(figsize=(10, 6))
ax1 = plt.gca()
ax1.plot(wide["Year"], wide["Total_Chargers_Lower"], marker="o", label="Lower Bound")
ax1.plot(wide["Year"], wide["Total_Chargers_Upper"], marker="o", label="Upper Bound")
ax1.plot(wide["Year"], wide["Forecast_Chargers"], linestyle="--", linewidth=2, label="Budget-Weighted Forecast (Monotone)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax1.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax1.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax1.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Chargers")
ax1.set_title("King County: EV Chargers — Lower vs Upper vs Budget-Weighted (Monotone)")
ax1.grid(True)
plt.tight_layout()
charger_chart = os.path.join(os.path.expanduser("~"), "Desktop", "charger_projection_mc_monotonic.png")
plt.savefig(charger_chart)
plt.close()

# --- B) EV Adoption Rate: Lower vs Upper vs Monotonic MC ---
plt.figure(figsize=(10, 6))
ax2 = plt.gca()
ax2.plot(wide["Year"], wide["Adoption_Rate_Lower"], marker="o", label="Lower Bound")
ax2.plot(wide["Year"], wide["Adoption_Rate_Upper"], marker="o", label="Upper Bound")
ax2.fill_between(wide["Year"], wide["Forecast_Adoption_P10"], wide["Forecast_Adoption_P90"],
                 alpha=0.20, label="MC Forecast Band (P10–P90)")
ax2.plot(wide["Year"], wide["Forecast_Adoption_P50"], linestyle="--", linewidth=2, label="MC Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax2.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax2.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax2.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax2.set_xlabel("Year")
ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("King County: EV Adoption — Lower vs Upper vs Monotonic MC (Growth-Responsive)")
ax2.grid(True)
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
adoption_chart = os.path.join(os.path.expanduser("~"), "Desktop", "ev_adoption_rate_mc_monotonic.png")
plt.savefig(adoption_chart)
plt.close()

# --- C) EV Registrations: MC P10/P50/P90 ---
plt.figure(figsize=(10, 6))
ax3 = plt.gca()
ax3.fill_between(wide["Year"], wide["Forecast_EVs_P10"], wide["Forecast_EVs_P90"],
                 alpha=0.20, label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"], wide["Forecast_EVs_P50"], linestyle="--", linewidth=2, label="MC EVs Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax3.axvspan(start, end, color=c, alpha=0.10)
ax3.legend(loc="best")
ax3.set_xlabel("Year")
ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("King County: EV Registrations — Monotonic MC P10 / P50 / P90")
ax3.grid(True)
plt.tight_layout()
evs_chart = os.path.join(os.path.expanduser("~"), "Desktop", "ev_registrations_mc_monotonic.png")
plt.savefig(evs_chart)
plt.close()

# ===================================
# Step 10: Embed charts back to Excel
# ===================================
wb = load_workbook(desktop_path)
ws = wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart), "A1")
ws.add_image(Image(adoption_chart), "A40")
ws.add_image(Image(evs_chart), "A79")
wb.save(desktop_path)

# ==============================
# Final sanity checks & summary
# ==============================
assert np.all(np.diff(wide["Forecast_Chargers"]) >= 0), "Final: chargers decreased — investigate!"
pct_within = 100.0 * wide["Forecast_within_bounds"].mean()
print(f"\n✅ Forecast complete. Excel and charts saved to: {desktop_path}")
print(f"• Median adoption within bounds in {pct_within:.1f}% of years.")
for yr in [2030, 2035, 2040, 2050]:
    r = wide.loc[wide['Year'] == yr].iloc[0]
    print(f"  - {yr}: P50={r['Forecast_Adoption_P50']:.2%}, "
          f"Lower={r['Adoption_Rate_Lower']:.2%}, Upper={r['Adoption_Rate_Upper']:.2%}, "
          f"EVs_P50={int(r['Forecast_EVs_P50']):,}, Chargers={int(r['Forecast_Chargers']):,}")


King County population (25-59): 1,241,805
Existing Superchargers: 13
Existing EVs (rows in EV file for King): 101838
Lower bound chargers (geo coverage): 184
Upper bound chargers (1 per 1,500 residents): 828

✅ Forecast complete. Excel and charts saved to: /Users/judycheng/Desktop/king_county_ev_projection_mc_monotonic.xlsx
• Median adoption within bounds in 80.8% of years.
  - 2030: P50=43.08%, Lower=22.25%, Upper=45.00%, EVs_P50=534,965, Chargers=297
  - 2035: P50=43.45%, Lower=35.62%, Upper=60.00%, EVs_P50=539,625, Chargers=297
  - 2040: P50=60.81%, Lower=49.00%, Upper=70.00%, EVs_P50=755,161, Chargers=300
  - 2050: P50=94.00%, Lower=95.00%, Upper=95.00%, EVs_P50=1,167,296, Chargers=538


In [7]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, math
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from matplotlib.ticker import PercentFormatter
import matplotlib.patches as mpatches

# =========================
# Paths (adjust if needed)
# =========================
super_path = "/Users/judycheng/Desktop/supercharger in washington state.xls"
residents_path = "/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx"
ev_path = "/Users/judycheng/Desktop/coordinates_output.xlsm"

# =========================
# Step 1: Read Excel files
# =========================
super_df = pd.read_excel(super_path)
residents_df = pd.read_excel(residents_path)
ev_df = pd.read_excel(ev_path)

# =========================
# Step 2: Pierce County data
# =========================
pierce_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "Pierce County")]
pierce_residents = residents_df[residents_df["County"] == "Pierce County"]
pierce_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "Pierce")]

population_column = "total"
population_25_59 = float(pierce_residents[population_column].values[0])
num_chargers = int(len(pierce_super))
num_evs = int(len(pierce_ev))

print(f"Pierce County population (25-59): {population_25_59:,.0f}")
print(f"Existing Superchargers (from Excel): {num_chargers}")
print(f"Existing EVs (rows): {num_evs}")

# =========================
# Step 3: Policy targets
# =========================
policy_targets = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}

# =========================
# Step 4: Charger targets
# =========================
pierce_county_area = 1790  # sq mi
radius_miles = 3
area_per_charger = np.pi * radius_miles**2  # ~28.27 sq mi
lower_chargers = int(np.ceil(pierce_county_area / area_per_charger))  # geo coverage (3-mile)
upper_base_target = population_25_59 / 2500.0                         # 1 per 2,500 residents

print(f"Lower bound chargers (geo coverage, 3-mile radius): {lower_chargers}")
print(f"Original upper bound (1/2,500 residents): {upper_base_target:.0f}")

# =========================
# Step 5: Build Lower/Upper scenarios
# =========================
years = list(range(2025, 2051))
results = []

# ---- LOWER (constant build toward coverage) ----
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers - num_chargers)
n_years = len(years)
yearly_build_lower_exact = total_lower_needed / n_years

print("\n--- LOWER BOUND DEBUG ---")
print(f"Existing chargers: {num_chargers}")
print(f"Target chargers:   {lower_chargers}")
print(f"Total add needed:  {total_lower_needed}")
print(f"Avg/yr (float):    {yearly_build_lower_exact:.2f}")
print("---------------------------\n")

for i, y in enumerate(years):
    new_lower = math.ceil(yearly_build_lower_exact)
    # ensure we don't overshoot the lower target; last year adjusts exactly
    if current_lower + new_lower > lower_chargers or i == len(years) - 1:
        new_lower = lower_chargers - current_lower
    new_lower = max(0, new_lower)
    current_lower += new_lower

    # Lower adoption path: modest P1; conservative P2; catch-up to 95% by 2050
    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5) * 0.35
    elif y <= 2040:
        eff_start = 0.10 + (policy_targets[2030] - 0.10) * 0.35
        eff = eff_start + (policy_targets[2040] * 0.70 - eff_start) * ((y - 2030) / 10)
    else:
        eff_start = policy_targets[2040] * 0.70
        eff = eff_start + (policy_targets[2050] - eff_start) * ((y - 2040) / 10)

    results.append({
        "Scenario": "Lower",
        "Year": y,
        "Total_Chargers": int(current_lower),
        "New_Chargers": int(new_lower),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

print(f"✅ Lower-bound chargers total at 2050: {int(current_lower)} (target: {lower_chargers})\n")

# ---- UPPER (front-load then taper) ----
base_build_rate = upper_base_target / n_years
current_upper = num_chargers

for y in years:
    if 2025 <= y <= 2030:
        yearly_add = int(round(base_build_rate * 1.0))
    elif 2031 <= y <= 2040:
        yearly_add = int(round(base_build_rate * 0.6))
    else:
        yearly_add = int(round(base_build_rate * 0.2))
    yearly_add = max(0, yearly_add)
    current_upper += yearly_add

    # Upper adoption tracks policy milestones
    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5)
    elif y <= 2035:
        eff = policy_targets[2030] + (policy_targets[2035] - policy_targets[2030]) * ((y - 2030) / 5)
    elif y <= 2040:
        eff = policy_targets[2035] + (policy_targets[2040] - policy_targets[2035]) * ((y - 2035) / 5)
    else:
        eff = policy_targets[2040] + (policy_targets[2050] - policy_targets[2040]) * ((y - 2040) / 10)

    results.append({
        "Scenario": "Upper",
        "Year": y,
        "Total_Chargers": int(current_upper),
        "New_Chargers": int(yearly_add),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

forecast_df = pd.DataFrame(results)

# =========================
# Step 6: Monte Carlo (monotone, charger-responsive)
# =========================
def clamp(x, lo=0.0, hi=0.9999): return float(np.clip(x, lo, hi))

yrs = np.array(years)
lo_path = forecast_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Lower"].values
hi_path = forecast_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Upper"].values

# --- 6A) Target path the MC reverts toward (not midpoint)
target_w_upper = []
for y in yrs:
    if 2025 <= y <= 2030:   target_w_upper.append(0.55)  # neutral to slightly upper-leaning
    elif 2031 <= y <= 2035: target_w_upper.append(0.25)  # lean low early P2
    else:                   target_w_upper.append(0.70)  # accelerate after 2035
target_w_upper = np.array(target_w_upper, dtype=float)

raw_target = (1 - target_w_upper) * lo_path + target_w_upper * hi_path
target_cap = 0.94  # keep <95%
target = np.minimum(raw_target, target_cap)
target = np.maximum.accumulate(target)  # target itself non-decreasing

# --- 6B) Public charger blend (monotone) + growth + momentum; home-share dampening
w_upper_charger = []
for y in yrs:
    if 2025 <= y <= 2030:   w_upper_charger.append(0.80)
    elif 2031 <= y <= 2040: w_upper_charger.append(0.30)
    else:                   w_upper_charger.append(0.55)
w_upper_charger = np.array(w_upper_charger, dtype=float)

lowC = forecast_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Lower"].values
hiC  = forecast_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Upper"].values

# Blended chargers and ENFORCE monotonicity
C_blend = ((1 - w_upper_charger) * lowC + w_upper_charger * hiC).round().astype(int)
C_blend = np.maximum.accumulate(C_blend)
assert np.all(np.diff(C_blend) >= 0), "Internal: C_blend decreased unexpectedly."

# Growth metrics
C_growth = np.r_[0, np.diff(C_blend)]
den = np.maximum(1.0, np.r_[C_blend[0], C_blend[:-1]])
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den>0)
C_growth_rate = np.clip(C_growth_rate, 0, 0.50)

# Home-charger share trajectory
home_share = []
for y in yrs:
    if 2025 <= y <= 2030:   home_share.append(0.55 + (0.72 - 0.55) * (y - 2025) / 5)
    elif 2031 <= y <= 2040: home_share.append(0.72 + (0.90 - 0.72) * (y - 2030) / 10)
    else:                   home_share.append(0.90 + (0.93 - 0.90) * (y - 2040) / 10)
home_share = np.array(home_share, dtype=float)

# Momentum: 3-year trailing average of growth rate
trail = 3
pad = np.full(trail - 1, C_growth_rate[1] if len(C_growth_rate) > 1 else 0.0)
C_gr_trailing = np.convolve(np.r_[pad, C_growth_rate], np.ones(trail)/trail, mode="valid")[:len(C_growth_rate)]
charger_momentum = np.clip(C_gr_trailing, 0.0, 0.40)

# =========================
# 6C) MC dynamics (monotone)
# =========================
N = 1000
T = len(yrs)
rng = np.random.default_rng(42)

# Phase-varying baselines
kappa_base = np.zeros(T)
beta_base  = np.zeros(T)
sigma      = np.zeros(T)
for i, y in enumerate(yrs):
    if 2025 <= y <= 2030:
        kappa_base[i] = 0.30; beta_base[i] = 0.45; sigma[i] = 0.015
    elif 2031 <= y <= 2035:
        kappa_base[i] = 0.25; beta_base[i] = 0.20; sigma[i] = 0.010
    elif 2036 <= y <= 2040:
        kappa_base[i] = 0.45; beta_base[i] = 0.30; sigma[i] = 0.018
    else:
        kappa_base[i] = 0.55; beta_base[i] = 0.30; sigma[i] = 0.020

# Adaptive charger coupling: damped by home_share, amplified by momentum
beta_effect = beta_base * (1 - home_share) * (1 + 1.5 * charger_momentum)
beta_effect = np.clip(beta_effect, 0.0, 0.9)

paths = np.zeros((N, T), dtype=float)
start_level = clamp(0.35 * lo_path[0] + 0.65 * hi_path[0], 0.0, target_cap)  # upper-leaning start (Phase 1 surge)
paths[:, 0] = start_level

for t in range(1, T):
    prev = paths[:, t-1]
    mean_revert = kappa_base[t] * (target[t] - prev)      # pull toward target (>=0 if below)
    charger_push = beta_effect[t] * C_growth_rate[t]      # >= 0
    noise = rng.normal(0.0, sigma[t], size=N)             # small noise
    delta = mean_revert + charger_push + noise
    delta = np.maximum(delta, 0.0)                        # no negative increments
    next_rate = prev + delta
    next_rate = np.maximum(next_rate, lo_path[t])         # never below Lower
    next_rate = np.minimum(next_rate, target_cap)         # keep < 95%
    next_rate = np.maximum(next_rate, prev)               # enforce monotonicity
    paths[:, t] = next_rate

# MC percentiles
p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p90 = np.percentile(paths, 90, axis=0)

ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# =========================
# Step 7: Build final table
# =========================
wide = forecast_df.pivot(index="Year", columns="Scenario", values=["Total_Chargers", "Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

# MC outputs
wide["Forecast_Adoption_P10"] = p10
wide["Forecast_Adoption_P50"] = p50
wide["Forecast_Adoption_P90"] = p90
wide["Forecast_EVs_P10"] = ev_p10
wide["Forecast_EVs_P50"] = ev_p50
wide["Forecast_EVs_P90"] = ev_p90

# Monotone, budget-weighted charger forecast
budget_weighted_C = C_blend.copy()
assert np.all(np.diff(budget_weighted_C) >= 0), "Forecast chargers decreased — investigate!"
wide["Forecast_Chargers"] = budget_weighted_C

# Optional: median within [Lower, Upper] band check
within_band = ((wide["Forecast_Adoption_P50"] >= wide["Adoption_Rate_Lower"]) &
               (wide["Forecast_Adoption_P50"] <= wide["Adoption_Rate_Upper"]))
wide["Forecast_within_bounds"] = within_band.astype(int)

# =========================
# Step 8: Save to Excel
# =========================
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "pierce_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path, engine="openpyxl") as writer:
    forecast_df.to_excel(writer, sheet_name="Scenarios_LU", index=False)
    wide.to_excel(writer, sheet_name="Forecast", index=False)

# =========================
# Step 9: Charts (phase-shaded)
# =========================
phase_spans = [(2025, 2030, "Phase 1"), (2030, 2040, "Phase 2"), (2040, 2050, "Phase 3")]
phase_colors = ["green", "yellow", "orange"]

# A) Chargers
plt.figure(figsize=(10,6))
ax1 = plt.gca()
ax1.plot(wide["Year"], wide["Total_Chargers_Lower"], marker="o", label="Lower Bound")
ax1.plot(wide["Year"], wide["Total_Chargers_Upper"], marker="o", label="Upper Bound")
ax1.plot(wide["Year"], wide["Forecast_Chargers"], linestyle="--", linewidth=2, label="Budget-Weighted Forecast (Monotone)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax1.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax1.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax1.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax1.set_xlabel("Year"); ax1.set_ylabel("Total Chargers")
ax1.set_title("Pierce County: EV Chargers — Lower vs Upper vs Forecast (Monotone)")
ax1.grid(True)
plt.tight_layout()
charger_chart = os.path.join(os.path.expanduser("~"), "Desktop", "pierce_charger_projection_mc_monotonic.png")
plt.savefig(charger_chart); plt.close()

# B) Adoption Rate
plt.figure(figsize=(10,6))
ax2 = plt.gca()
ax2.plot(wide["Year"], wide["Adoption_Rate_Lower"], marker="o", label="Lower Bound")
ax2.plot(wide["Year"], wide["Adoption_Rate_Upper"], marker="o", label="Upper Bound")
ax2.fill_between(wide["Year"], wide["Forecast_Adoption_P10"], wide["Forecast_Adoption_P90"], alpha=0.20, label="MC Band (P10–P90)")
ax2.plot(wide["Year"], wide["Forecast_Adoption_P50"], linestyle="--", linewidth=2, label="MC Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax2.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax2.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax2.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax2.set_xlabel("Year"); ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("Pierce County: EV Adoption — Lower vs Upper vs Monotonic MC (Growth-Responsive)")
ax2.grid(True); ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
adoption_chart = os.path.join(os.path.expanduser("~"), "Desktop", "pierce_ev_adoption_rate_mc_monotonic.png")
plt.savefig(adoption_chart); plt.close()

# C) EV Registrations
plt.figure(figsize=(10,6))
ax3 = plt.gca()
ax3.fill_between(wide["Year"], wide["Forecast_EVs_P10"], wide["Forecast_EVs_P90"], alpha=0.20, label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"], wide["Forecast_EVs_P50"], linestyle="--", linewidth=2, label="MC EVs Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax3.axvspan(start, end, color=c, alpha=0.10)
ax3.legend(loc="best")
ax3.set_xlabel("Year"); ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("Pierce County: EV Registrations — Monotonic MC P10 / P50 / P90")
ax3.grid(True)
plt.tight_layout()
evs_chart = os.path.join(os.path.expanduser("~"), "Desktop", "pierce_ev_registrations_mc_monotonic.png")
plt.savefig(evs_chart); plt.close()

# =========================
# Step 10: Embed charts in Excel
# =========================
wb = load_workbook(desktop_path)
ws = wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart), "A1")
ws.add_image(Image(adoption_chart), "A40")
ws.add_image(Image(evs_chart), "A79")
wb.save(desktop_path)

# =========================
# Final checks & summary
# =========================
assert np.all(np.diff(wide["Forecast_Chargers"]) >= 0), "Final: chargers decreased — investigate!"
pct_within = 100.0 * wide["Forecast_within_bounds"].mean()
print(f"\n✅ Projection complete. Excel & charts saved to: {desktop_path}")
print(f"• Median adoption within bounds in {pct_within:.1f}% of years.")
print(f"• Final 2050 Upper chargers (running total): {int(forecast_df[forecast_df['Scenario']=='Upper'].iloc[-1]['Total_Chargers'])}")
print(f"• 2050 P50 adoption: {p50[-1]:.2%} (capped <95%)")


Pierce County population (25-59): 448,201
Existing Superchargers (from Excel): 1
Existing EVs (rows): 16277
Lower bound chargers (geo coverage, 3-mile radius): 64
Original upper bound (1/2,500 residents): 179

--- LOWER BOUND DEBUG ---
Existing chargers: 1
Target chargers:   64
Total add needed:  63
Avg/yr (float):    2.42
---------------------------

✅ Lower-bound chargers total at 2050: 64 (target: 64)


✅ Projection complete. Excel & charts saved to: /Users/judycheng/Desktop/pierce_county_ev_projection_mc_monotonic.xlsx
• Median adoption within bounds in 80.8% of years.
• Final 2050 Upper chargers (running total): 93
• 2050 P50 adoption: 94.00% (capped <95%)


In [11]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, math
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from matplotlib.ticker import PercentFormatter
import matplotlib.patches as mpatches

# =========================
# Step 1: Read Excel files
# =========================
super_df = pd.read_excel("/Users/judycheng/Desktop/supercharger in washington state.xls")
residents_df = pd.read_excel("/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx")
ev_df = pd.read_excel("/Users/judycheng/Desktop/coordinates_output.xlsm")

# =========================
# Step 2: Kitsap County data
# =========================
kitsap_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "Kitsap County")]
kitsap_residents = residents_df[residents_df["County"] == "Kitsap County"]
kitsap_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "Kitsap")]

population_column = "total"
population_25_59 = float(kitsap_residents[population_column].values[0])
num_chargers = int(len(kitsap_super))
num_evs = int(len(kitsap_ev))

print(f"Kitsap County population (25–59): {population_25_59:,.0f}")
print(f"Existing Superchargers: {num_chargers}")
print(f"Existing EVs: {num_evs}")

# =========================
# Step 3: Policy targets
# =========================
policy_targets = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}

# =========================
# Step 4: Charger targets
# =========================
kitsap_county_area = 566  # sq mi
radius_miles = 5
area_per_charger = np.pi * radius_miles**2  # ~78.54 sq mi per charger
lower_chargers = int(np.ceil(kitsap_county_area / area_per_charger))
upper_base_target = population_25_59 / 2500.0  # 1 per 2,500 residents

print(f"Lower bound chargers (5-mile coverage): {lower_chargers}")
print(f"Upper bound (1 per 2,500 residents): {upper_base_target:.0f}")

# =========================
# Step 5: Build Lower/Upper scenarios
# =========================
years = list(range(2025, 2051))
results = []
n_years = len(years)

# ---- LOWER ----
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers - num_chargers)
yearly_build_lower_exact = total_lower_needed / n_years

for i, y in enumerate(years):
    new_lower = math.ceil(yearly_build_lower_exact)
    if current_lower + new_lower > lower_chargers or i == len(years) - 1:
        new_lower = lower_chargers - current_lower
    current_lower += new_lower

    # Conservative adoption path
    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5) * 0.35
    elif y <= 2040:
        eff_start = 0.10 + (policy_targets[2030] - 0.10) * 0.35
        eff = eff_start + (policy_targets[2040]*0.70 - eff_start)*((y - 2030)/10)
    else:
        eff_start = policy_targets[2040]*0.70
        eff = eff_start + (policy_targets[2050]-eff_start)*((y - 2040)/10)

    results.append({"Scenario":"Lower","Year":y,
                    "Total_Chargers":int(current_lower),
                    "New_Chargers":int(new_lower),
                    "Adoption_Rate":float(np.clip(eff,0.0,0.9999))})

# ---- UPPER ----
base_build_rate = upper_base_target / n_years
current_upper = num_chargers
for y in years:
    if 2025 <= y <= 2030: yearly_add = int(round(base_build_rate * 1.0))
    elif 2031 <= y <= 2040: yearly_add = int(round(base_build_rate * 0.6))
    else: yearly_add = int(round(base_build_rate * 0.2))
    current_upper += yearly_add

    if y <= 2030:
        eff = 0.10 + (policy_targets[2030]-0.10)*((y-2025)/5)
    elif y <= 2035:
        eff = policy_targets[2030]+(policy_targets[2035]-policy_targets[2030])*((y-2030)/5)
    elif y <= 2040:
        eff = policy_targets[2035]+(policy_targets[2040]-policy_targets[2035])*((y-2035)/5)
    else:
        eff = policy_targets[2040]+(policy_targets[2050]-policy_targets[2040])*((y-2040)/10)

    results.append({"Scenario":"Upper","Year":y,
                    "Total_Chargers":int(current_upper),
                    "New_Chargers":yearly_add,
                    "Adoption_Rate":float(np.clip(eff,0.0,0.9999))})

forecast_df = pd.DataFrame(results)

# =========================
# Step 6: Monte Carlo (monotone + charger momentum)
# =========================
def clamp(x, lo=0.0, hi=0.9999): 
    return float(np.clip(x, lo, hi))

yrs = np.array(years)
lo_path = forecast_df.pivot(index="Year",columns="Scenario",values="Adoption_Rate")["Lower"].values
hi_path = forecast_df.pivot(index="Year",columns="Scenario",values="Adoption_Rate")["Upper"].values

# 6A) Target path
target_w_upper = np.where(yrs<=2030,0.55,np.where(yrs<=2035,0.25,0.70))
raw_target = (1-target_w_upper)*lo_path + target_w_upper*hi_path
target_cap = 0.94
target = np.minimum(raw_target,target_cap)
target = np.maximum.accumulate(target)

# 6B) Chargers + home charging
w_upper_charger = np.where(yrs<=2030,0.80,np.where(yrs<=2040,0.30,0.55))
lowC = forecast_df.pivot(index="Year",columns="Scenario",values="Total_Chargers")["Lower"].values
hiC  = forecast_df.pivot(index="Year",columns="Scenario",values="Total_Chargers")["Upper"].values
C_blend = ((1-w_upper_charger)*lowC + w_upper_charger*hiC).round().astype(int)
C_blend = np.maximum.accumulate(C_blend)  # ✅ monotonic chargers
assert np.all(np.diff(C_blend)>=0), "Chargers decreased!"

# FIXED: Safe float division for growth rate
C_growth = np.r_[0, np.diff(C_blend)].astype(float)
den = np.maximum(1.0, np.r_[C_blend[0], C_blend[:-1]]).astype(float)
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den>0)
C_growth_rate = np.clip(C_growth_rate, 0, 0.5)

# Home-charging fraction
home_share = np.where(yrs<=2030,
                      0.55+(0.72-0.55)*(yrs-2025)/5,
                      np.where(yrs<=2040,
                               0.72+(0.90-0.72)*(yrs-2030)/10,
                               0.90+(0.93-0.90)*(yrs-2040)/10))

# Charger momentum: 3-year trailing average
trail = 3
pad = np.full(trail-1, C_growth_rate[1] if len(C_growth_rate)>1 else 0.0)
C_gr_trailing = np.convolve(np.r_[pad, C_growth_rate], np.ones(trail)/trail, mode="valid")[:len(C_growth_rate)]
charger_momentum = np.clip(C_gr_trailing, 0, 0.4)

# 6C) MC evolution
N, T = 1000, len(yrs)
rng = np.random.default_rng(42)
kappa = np.zeros(T); beta = np.zeros(T); sigma = np.zeros(T)

for i,y in enumerate(yrs):
    if 2025<=y<=2030: kappa[i]=0.30; beta[i]=0.45; sigma[i]=0.015
    elif 2031<=y<=2035: kappa[i]=0.25; beta[i]=0.20; sigma[i]=0.010
    elif 2036<=y<=2040: kappa[i]=0.45; beta[i]=0.30; sigma[i]=0.018
    else: kappa[i]=0.55; beta[i]=0.30; sigma[i]=0.020

beta_effect = beta * (1 - home_share) * (1 + 1.5 * charger_momentum)
beta_effect = np.clip(beta_effect, 0, 0.9)

paths = np.zeros((N, T))
start_level = clamp(0.35 * lo_path[0] + 0.65 * hi_path[0], 0.0, target_cap)
paths[:,0] = start_level

for t in range(1, T):
    prev = paths[:, t-1]
    mean_revert = kappa[t] * (target[t] - prev)
    charger_push = beta_effect[t] * C_growth_rate[t]
    noise = rng.normal(0.0, sigma[t], size=N)
    delta = np.maximum(mean_revert + charger_push + noise, 0.0)
    next_rate = np.minimum(np.maximum(prev + delta, lo_path[t]), target_cap)
    next_rate = np.maximum(next_rate, prev)
    paths[:,t] = next_rate

p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p90 = np.percentile(paths, 90, axis=0)
ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# =========================
# Step 7: Assemble + save
# =========================
wide = forecast_df.pivot(index="Year",columns="Scenario",values=["Total_Chargers","Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a,b in wide.columns]
wide = wide.reset_index()
wide["Forecast_Adoption_P10"]=p10
wide["Forecast_Adoption_P50"]=p50
wide["Forecast_Adoption_P90"]=p90
wide["Forecast_EVs_P10"]=ev_p10
wide["Forecast_EVs_P50"]=ev_p50
wide["Forecast_EVs_P90"]=ev_p90
wide["Forecast_Chargers"]=C_blend

desktop_path = os.path.join(os.path.expanduser("~"),"Desktop","kitsap_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path,engine="openpyxl") as writer:
    forecast_df.to_excel(writer,sheet_name="Scenarios_LU",index=False)
    wide.to_excel(writer,sheet_name="Forecast",index=False)

# =========================
# Step 8: Charts
# =========================
phase_spans=[(2025,2030,"Phase 1"),(2030,2040,"Phase 2"),(2040,2050,"Phase 3")]
phase_colors=["green","yellow","orange"]

# Chargers
plt.figure(figsize=(10,6))
ax1=plt.gca()
ax1.plot(wide["Year"],wide["Total_Chargers_Lower"],marker="o",label="Lower Bound")
ax1.plot(wide["Year"],wide["Total_Chargers_Upper"],marker="o",label="Upper Bound")
ax1.plot(wide["Year"],wide["Forecast_Chargers"],"--",linewidth=2,label="Forecast (Monotone)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax1.axvspan(s,e,color=c,alpha=0.10)
ax1.legend(loc="best")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Chargers")
ax1.set_title("Kitsap County: EV Chargers — Lower / Upper / Monotone Forecast")
ax1.grid(True)
plt.tight_layout()
charger_chart=os.path.join(os.path.expanduser("~"),"Desktop","kitsap_charger_projection_mc_monotonic.png")
plt.savefig(charger_chart)
plt.close()

# Adoption
plt.figure(figsize=(10,6))
ax2=plt.gca()
ax2.plot(wide["Year"],wide["Adoption_Rate_Lower"],marker="o",label="Lower Bound")
ax2.plot(wide["Year"],wide["Adoption_Rate_Upper"],marker="o",label="Upper Bound")
ax2.fill_between(wide["Year"],wide["Forecast_Adoption_P10"],wide["Forecast_Adoption_P90"],alpha=0.20,label="MC Band P10–P90")
ax2.plot(wide["Year"],wide["Forecast_Adoption_P50"],"--",linewidth=2,label="MC Median (P50)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax2.axvspan(s,e,color=c,alpha=0.10)
ax2.legend(loc="best")
ax2.set_xlabel("Year")
ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("Kitsap County: EV Adoption — Monotone MC (Growth-Responsive)")
ax2.grid(True)
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
adoption_chart=os.path.join(os.path.expanduser("~"),"Desktop","kitsap_ev_adoption_rate_mc_monotonic.png")
plt.savefig(adoption_chart)
plt.close()

# Registrations
plt.figure(figsize=(10,6))
ax3=plt.gca()
ax3.fill_between(wide["Year"],wide["Forecast_EVs_P10"],wide["Forecast_EVs_P90"],alpha=0.20,label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"],wide["Forecast_EVs_P50"],"--",linewidth=2,label="MC EVs Median (P50)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax3.axvspan(s,e,color=c,alpha=0.10)
ax3.legend(loc="best")
ax3.set_xlabel("Year")
ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("Kitsap County: EV Registrations — Monotonic MC P10 / P50 / P90")
ax3.grid(True)
plt.tight_layout()
evs_chart=os.path.join(os.path.expanduser("~"),"Desktop","kitsap_ev_registrations_mc_monotonic.png")
plt.savefig(evs_chart)
plt.close()

# =========================
# Step 9: Embed charts in Excel
# =========================
wb=load_workbook(desktop_path)
ws=wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart),"A1")
ws.add_image(Image(adoption_chart),"A40")
ws.add_image(Image(evs_chart),"A79")
wb.save(desktop_path)

# =========================
# Final summary
# =========================
assert np.all(np.diff(wide["Forecast_Chargers"])>=0),"Final forecast not monotonic!"
print(f"\n✅ Kitsap projection complete: {desktop_path}")
print(f"2050 P50 adoption: {p50[-1]:.2%} (capped <95%)")


Kitsap County population (25–59): 125,820
Existing Superchargers: 0
Existing EVs: 6428
Lower bound chargers (5-mile coverage): 8
Upper bound (1 per 2,500 residents): 50

✅ Kitsap projection complete: /Users/judycheng/Desktop/kitsap_county_ev_projection_mc_monotonic.xlsx
2050 P50 adoption: 94.00% (capped <95%)


In [12]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, math
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
import matplotlib.patches as mpatches
from matplotlib.ticker import PercentFormatter

# =========================
# Step 1: Read Excel files
# =========================
super_df = pd.read_excel("/Users/judycheng/Desktop/supercharger in washington state.xls")
residents_df = pd.read_excel("/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx")
ev_df = pd.read_excel("/Users/judycheng/Desktop/coordinates_output.xlsm")

# =========================
# Step 2: Chelan County data
# =========================
chelan_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "Chelan County")]
chelan_residents = residents_df[residents_df["County"] == "Chelan County"]
chelan_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "Chelan")]

population_column = "total"
population_25_59 = float(chelan_residents[population_column].values[0])
num_chargers = int(len(chelan_super))
num_evs = int(len(chelan_ev))

print(f"Chelan County population (25–59): {population_25_59:,.0f}")
print(f"Existing Superchargers: {num_chargers}")
print(f"Existing EVs: {num_evs}")

# =========================
# Step 3: Policy targets
# =========================
policy_targets = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}

# =========================
# Step 4: Charger targets
# =========================
county_area = 2965     # sq mi
radius_miles = 15
lower_chargers = math.ceil(county_area / (np.pi * radius_miles**2))  # ~15-mi radius coverage
upper_chargers = math.ceil(population_25_59 / 5000)                  # 1 per 5,000 residents

print(f"Lower bound chargers (15-mi coverage): {lower_chargers}")
print(f"Upper bound chargers (1 per 5,000 residents): {upper_chargers}")

# =========================
# Step 5: Build Lower / Upper scenarios
# =========================
years = list(range(2025, 2051))
results = []

# ---- LOWER: steady build toward geo-coverage; conservative adoption in Phase 2 ----
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers - num_chargers)
n_years = len(years)
yearly_build_lower = int(round(total_lower_needed / n_years)) if n_years > 0 else 0
yearly_build_lower = max(1, yearly_build_lower) if total_lower_needed > 0 else 0

for i, y in enumerate(years):
    new_lower = yearly_build_lower if current_lower < lower_chargers else 0
    if current_lower + new_lower > lower_chargers:
        new_lower = lower_chargers - current_lower
    current_lower += max(0, new_lower)

    if y <= 2030:
        eff = 0.10 + (policy_targets[2030]-0.10) * ((y-2025)/5) * 0.35
    elif y <= 2040:
        eff_start = 0.10 + (policy_targets[2030]-0.10) * 0.35
        eff = eff_start + (policy_targets[2040]*0.70 - eff_start) * ((y-2030)/10)  # lean below Upper
    else:
        eff_start = policy_targets[2040]*0.70
        eff = eff_start + (policy_targets[2050]-eff_start) * ((y-2040)/10)         # catch-up to 95%

    results.append({
        "Scenario":"Lower","Year":y,
        "Total_Chargers":int(current_lower),
        "New_Chargers":int(max(0,new_lower)),
        "Adoption_Rate":float(np.clip(eff,0.0,0.9999)),
        "EVs":int(population_25_59 * np.clip(eff,0.0,0.9999))
    })

# ---- UPPER: smooth path to population target; adoption follows policy milestones ----
current_upper = num_chargers
for y in years:
    remaining = max(0, upper_chargers - current_upper)
    remaining_years = max(1, 2050 - y + 1)
    yearly_add = int(math.ceil(remaining / remaining_years)) if remaining > 0 else 0
    current_upper += yearly_add

    if y <= 2030:
        eff = 0.10 + (policy_targets[2030]-0.10)*((y-2025)/5)
    elif y <= 2035:
        eff = policy_targets[2030] + (policy_targets[2035]-policy_targets[2030])*((y-2030)/5)
    elif y <= 2040:
        eff = policy_targets[2035] + (policy_targets[2040]-policy_targets[2035])*((y-2035)/5)
    else:
        eff = policy_targets[2040] + (policy_targets[2050]-policy_targets[2040])*((y-2040)/10)

    results.append({
        "Scenario":"Upper","Year":y,
        "Total_Chargers":int(current_upper),
        "New_Chargers":int(yearly_add),
        "Adoption_Rate":float(np.clip(eff,0.0,0.9999)),
        "EVs":int(population_25_59 * np.clip(eff,0.0,0.9999))
    })

sc_df = pd.DataFrame(results)

# =========================
# Step 6: Monte Carlo (monotone + charger momentum)
# =========================
def clamp(x, lo=0.0, hi=0.9999):
    return float(np.clip(x, lo, hi))

yrs = sc_df["Year"].unique()
lo_path = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Lower"].values
hi_path = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Upper"].values

# 6A) Target path (not midpoint): weights vary by phase; cap < 95%
target_w_upper = []
for y in yrs:
    if 2025 <= y <= 2030:   target_w_upper.append(0.55)  # Phase 1: slight upper-lean
    elif 2031 <= y <= 2035: target_w_upper.append(0.25)  # Early Phase 2: lean low
    else:                   target_w_upper.append(0.70)  # Late Phase 2 & Phase 3: accelerate
target_w_upper = np.array(target_w_upper, dtype=float)

raw_target = (1 - target_w_upper) * lo_path + target_w_upper * hi_path
target_cap = 0.94
target = np.minimum(raw_target, target_cap)
target = np.maximum.accumulate(target)  # ensure non-decreasing target

# 6B) Charger blend (monotone), growth rate (float-safe), home-share, momentum
w_upper_charger = []
for y in yrs:
    if 2025 <= y <= 2030:   w_upper_charger.append(0.80)
    elif 2031 <= y <= 2040: w_upper_charger.append(0.30)
    else:                   w_upper_charger.append(0.55)
w_upper_charger = np.array(w_upper_charger, dtype=float)

lowC = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Lower"].values
hiC  = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Upper"].values

# ✅ Enforce monotonic chargers for the blended "forecast" path
C_blend = ((1 - w_upper_charger) * lowC + w_upper_charger * hiC).round().astype(int)
C_blend = np.maximum.accumulate(C_blend)
assert np.all(np.diff(C_blend) >= 0), "Internal: blended charger path decreased!"

# Float-safe charger growth rate
C_growth = np.r_[0, np.diff(C_blend)].astype(float)
den = np.maximum(1.0, np.r_[C_blend[0], C_blend[:-1]]).astype(float)
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den > 0)
C_growth_rate = np.clip(C_growth_rate, 0, 0.50)

# Home-charging share (rural tilt)
home_share = []
for y in yrs:
    if 2025 <= y <= 2030:   home_share.append(0.50 + (0.70 - 0.50) * (y - 2025) / 5)  # 50%→70%
    elif 2031 <= y <= 2040: home_share.append(0.70 + (0.88 - 0.70) * (y - 2030) / 10) # 70%→88%
    else:                   home_share.append(0.88 + (0.92 - 0.88) * (y - 2040) / 10) # 88%→92%
home_share = np.array(home_share, dtype=float)

# Charger momentum: 3-year trailing average of growth
trail = 3
pad = np.full(trail - 1, C_growth_rate[1] if len(C_growth_rate) > 1 else 0.0)
C_gr_trailing = np.convolve(np.r_[pad, C_growth_rate], np.ones(trail)/trail, mode="valid")[:len(C_growth_rate)]
charger_momentum = np.clip(C_gr_trailing, 0.0, 0.40)

# 6C) MC evolution (monotone)
N = 1000
T = len(yrs)
rng = np.random.default_rng(42)

kappa = np.zeros(T); beta = np.zeros(T); sigma = np.zeros(T)
for i, y in enumerate(yrs):
    if 2025 <= y <= 2030:   kappa[i]=0.30; beta[i]=0.45; sigma[i]=0.015
    elif 2031 <= y <= 2035: kappa[i]=0.25; beta[i]=0.20; sigma[i]=0.010
    elif 2036 <= y <= 2040: kappa[i]=0.45; beta[i]=0.30; sigma[i]=0.018
    else:                   kappa[i]=0.55; beta[i]=0.30; sigma[i]=0.020

# Couple adoption to charger growth, damped by home_share and boosted by momentum
beta_effect = beta * (1 - home_share) * (1 + 1.5 * charger_momentum)
beta_effect = np.clip(beta_effect, 0.0, 0.90)

paths = np.zeros((N, T), dtype=float)
start_level = clamp(0.35 * lo_path[0] + 0.65 * hi_path[0], 0.0, target_cap)  # rural corridor uplift early
paths[:, 0] = start_level

for t in range(1, T):
    prev = paths[:, t-1]
    mean_revert = kappa[t] * (target[t] - prev)
    charger_push = beta_effect[t] * C_growth_rate[t]
    noise = rng.normal(0.0, sigma[t], size=N)
    delta = np.maximum(mean_revert + charger_push + noise, 0.0)   # forbid negative increments
    next_rate = prev + delta
    next_rate = np.maximum(next_rate, lo_path[t])                  # never below Lower path that year
    next_rate = np.minimum(next_rate, target_cap)                  # keep < 95%
    next_rate = np.maximum(next_rate, prev)                        # enforce monotonicity
    paths[:, t] = next_rate

# MC percentiles and EV counts
p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p90 = np.percentile(paths, 90, axis=0)
ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# =========================
# Step 7: Save results with forecast columns
# =========================
wide = sc_df.pivot(index="Year", columns="Scenario", values=["Total_Chargers", "Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

wide["Forecast_Adoption_P10"] = p10
wide["Forecast_Adoption_P50"] = p50
wide["Forecast_Adoption_P90"] = p90
wide["Forecast_EVs_P10"] = ev_p10
wide["Forecast_EVs_P50"] = ev_p50
wide["Forecast_EVs_P90"] = ev_p90

# ✅ Use the monotone blended charger path as the forecast
wide["Forecast_Chargers"] = C_blend
assert np.all(np.diff(wide["Forecast_Chargers"]) >= 0), "Final: charger forecast not monotone!"

desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path, engine="openpyxl") as writer:
    sc_df.to_excel(writer, sheet_name="Scenarios_LU", index=False)
    wide.to_excel(writer, sheet_name="Forecast", index=False)

# =========================
# Step 8: Charts (with phase shading)
# =========================
phase_spans = [(2025, 2030, "Phase 1"), (2030, 2040, "Phase 2"), (2040, 2050, "Phase 3")]
phase_colors = ["green", "yellow", "orange"]

# 1) Chargers
plt.figure(figsize=(10,6))
ax1 = plt.gca()
ax1.plot(wide["Year"], wide["Total_Chargers_Lower"], marker="o", label="Lower Bound")
ax1.plot(wide["Year"], wide["Total_Chargers_Upper"], marker="o", label="Upper Bound")
ax1.plot(wide["Year"], wide["Forecast_Chargers"], "--", linewidth=2, label="Forecast (Monotone)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax1.axvspan(s,e,color=c,alpha=0.10)
ax1.legend(loc="best")
ax1.set_xlabel("Year"); ax1.set_ylabel("Total Chargers")
ax1.set_title("Chelan County: EV Chargers — Lower / Upper / Monotone Forecast")
ax1.grid(True)
plt.tight_layout()
charger_chart = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_chargers_mc_monotonic.png")
plt.savefig(charger_chart); plt.close()

# 2) Adoption Rate
plt.figure(figsize=(10,6))
ax2 = plt.gca()
ax2.plot(wide["Year"], wide["Adoption_Rate_Lower"], marker="o", label="Lower Bound")
ax2.plot(wide["Year"], wide["Adoption_Rate_Upper"], marker="o", label="Upper Bound")
ax2.fill_between(wide["Year"], wide["Forecast_Adoption_P10"], wide["Forecast_Adoption_P90"], alpha=0.20, label="MC Band (P10–P90)")
ax2.plot(wide["Year"], wide["Forecast_Adoption_P50"], "--", linewidth=2, label="MC Median (P50)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax2.axvspan(s,e,color=c,alpha=0.10)
ax2.legend(loc="best")
ax2.set_xlabel("Year"); ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("Chelan County: EV Adoption — Monotone MC (Growth-Responsive)")
ax2.grid(True); ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
adoption_chart = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_adoption_mc_monotonic.png")
plt.savefig(adoption_chart); plt.close()

# 3) EV Registrations
plt.figure(figsize=(10,6))
ax3 = plt.gca()
ax3.fill_between(wide["Year"], wide["Forecast_EVs_P10"], wide["Forecast_EVs_P90"], alpha=0.20, label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"], wide["Forecast_EVs_P50"], "--", linewidth=2, label="MC EVs Median (P50)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax3.axvspan(s,e,color=c,alpha=0.10)
ax3.legend(loc="best")
ax3.set_xlabel("Year"); ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("Chelan County: EV Registrations — Monotone MC P10 / P50 / P90")
ax3.grid(True)
plt.tight_layout()
evs_chart = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_evs_mc_monotonic.png")
plt.savefig(evs_chart); plt.close()

# =========================
# Step 9: Embed charts in Excel
# =========================
wb = load_workbook(desktop_path)
ws = wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart), "A1")
ws.add_image(Image(adoption_chart), "A40")
ws.add_image(Image(evs_chart), "A79")
wb.save(desktop_path)

# =========================
# Summary
# =========================
print(f"\n✅ Chelan County EV Projection saved to: {desktop_path}")
print(f"2050 P50 adoption (capped <95%): {p50[-1]:.2%}")


Chelan County population (25–59): 33,879
Existing Superchargers: 3
Existing EVs: 1250
Lower bound chargers (15-mi coverage): 5
Upper bound chargers (1 per 5,000 residents): 7

✅ Chelan County EV Projection saved to: /Users/judycheng/Desktop/chelan_county_ev_projection_mc_monotonic.xlsx
2050 P50 adoption (capped <95%): 94.00%
